In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
hfit = load_physics_classes()

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/"
#file = "TLE_1e20_pion_all_update_calo.df"
file = "TLE_1e20_pion_stopping_update_calo.df"

mc_bnb_df = load_df(bnb_path + file, keys2load, 4)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))

In [ ]:

pdg = 13
if "pion" in file:
    pdg = 211        
use_data = "data" in file

    
theoretical_mpv = make_theoretical_mpv_func(hfit, pdg=pdg) 
conv_tf1_map         = hfit.get_conv_function_map(pdg, "none", 500, use_data)
conv_shifted_tf1_map = hfit.get_conv_function_map(pdg, "shift", 500, use_data)
langau_map = hfit.get_langau_map(pdg, 500, use_data)

In [ ]:
def get_conv_tf1_for_slice(conv_map, plane, rr_center, max_rr):
    """Look up the TF1* in a conv_tf1_map / conv_shifted_tf1_map for the
    residual-range bin closest to rr_center. Bins are 1 cm wide, centered
    at i_rr + 0.5 for i_rr = 0 .. max_rr-1 (see get_conv_function_map).
    Returns None if plane is missing or the index falls outside the map.
    """
    if plane not in conv_map:
        return None
    i_rr = int(np.floor(rr_center))   # rr_center == i_rr + 0.5 for an aligned bin
    if i_rr < 0 or i_rr >= max_rr or i_rr >= len(conv_map[plane]):
        return None
    return conv_map[plane][i_rr]

In [ ]:
def robust_max_x(f, xmin, xmax, n_scan=10000):
    """Python port of Hypfit::robust_max_x -- identical grid-scan +
    parabolic-refinement algorithm, so comparisons against C++-derived
    peaks (PDF_max, f0_max_x) aren't confounded by using a different
    optimizer (ROOT's GetMaximumX uses Brent's method internally, which
    can converge to a slightly different x than a grid+parabola search
    even on the same function/domain)."""
    step = (xmax - xmin) / (n_scan - 1)
    ys = np.array([f.Eval(xmin + i * step) for i in range(n_scan)])
    best_i = int(np.argmax(ys))
    best_x = xmin + best_i * step
    if best_i <= 0 or best_i >= n_scan - 1:
        return best_x
    y0, y1, y2 = ys[best_i - 1], ys[best_i], ys[best_i + 1]
    denom = y0 - 2.0 * y1 + y2
    if denom == 0:
        return best_x
    delta = 0.5 * (y0 - y2) / denom
    return best_x + delta * step

def plot_slice_diagnostic(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit=None,
    pdg=13,
    mass=None,
    dedx_col="dedx",
    nbins=50,
    hist_xmin=0.0,
    hist_xmax=20.0,
    first_stage_range=(0.0, 10.0),
    conv_tf1_map=None,
    conv_shifted_tf1_map=None,
    langau_map=None,
    conv_max_rr=1000,
    x_limits=None,
    plot_only_theory=False,
):
    # If x_limits is specified, use it to redefine the histogram boundaries
    if x_limits is not None:
        hist_xmin, hist_xmax = x_limits

    sl = df if tpc == -1 else df[df["tpc"] == tpc]
    sl = sl[(sl["rr"] >= rr_min) & (sl["rr"] < rr_max) & (sl["pitch"] <= 2)]
    rr_center = 0.5 * (rr_min + rr_max)

    i_rr_lookup = int(np.floor(rr_center))
    i_rr_lookup = max(0, min(i_rr_lookup, conv_max_rr - 1))
    rr_aligned = i_rr_lookup + 0.5

    # If plot_only_theory is True, disable all convolution/map fits
    if plot_only_theory:
        conv_tf1_map = None
        conv_shifted_tf1_map = None
        langau_map = None

    fig, ax = plt.subplots(figsize=(8, 5.5))
    counts, edges, _ = ax.hist(
        sl[dedx_col].dropna(),
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"Measured PDF (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 400)
    data_max = counts.max() if len(counts) else 0.0
    curve_max = data_max

    # Track curves for visible range scaling
    y_theo, y_conv, y_conv_shifted, y_langau_map = None, None, None, None

    hist = th1_from_series(
        sl[dedx_col],
        f"diag_p{plane}_t{tpc}",
        "",
        nbins,
        hist_xmin,
        hist_xmax,
    )

    if hfit is not None:
        mean_pitch = 0.32
        pdf = build_theoretical_pdf(hfit, pdg, rr_aligned, mean_pitch, mass=mass)
        norm = len(sl) * bin_width
        y_theo = np.array([pdf.Eval(xv) * norm for xv in x_eval])
        if y_theo.max() > 0 and data_max > 0:
            y_theo = y_theo / y_theo.max() * data_max
        ax.plot(
            x_eval,
            y_theo,
            color="#785EF0",  # Colorblind-friendly purple
            linestyle=":",
            lw=2.5,
            label="Theoretical PDF",
        )
        curve_max = max(curve_max, y_theo.max())

    if conv_tf1_map is not None:
        f_conv = get_conv_tf1_for_slice(conv_tf1_map, plane, rr_center, conv_max_rr)
        if f_conv is not None:
            y_conv = np.array([f_conv.Eval(xv) for xv in x_eval])
            if y_conv.max() > 0 and data_max > 0:
                y_conv = y_conv / y_conv.max() * data_max
            ax.plot(
                x_eval,
                y_conv,
                color="crimson",
                linestyle="-.",
                lw=2.0,
                label="Convolution",
            )
            curve_max = max(curve_max, y_conv.max())

    if conv_shifted_tf1_map is not None:
        f_conv_shifted = get_conv_tf1_for_slice(
            conv_shifted_tf1_map, plane, rr_center, conv_max_rr
        )
        if f_conv_shifted is not None:
            y_conv_shifted = np.array([f_conv_shifted.Eval(xv) for xv in x_eval])
            if y_conv_shifted.max() > 0 and data_max > 0:
                y_conv_shifted = y_conv_shifted / y_conv_shifted.max() * data_max
            ax.plot(
                x_eval,
                y_conv_shifted,
                color="teal",
                linestyle="-.",
                lw=2.0,
                label="Convolution (shifted)",
            )
            curve_max = max(curve_max, y_conv_shifted.max())

    if langau_map is not None:
        f_langau_map = get_conv_tf1_for_slice(langau_map, plane, rr_center, conv_max_rr)
        if f_langau_map is not None:
            y_langau_map = np.array([f_langau_map.Eval(xv) for xv in x_eval])
            if y_langau_map.max() > 0 and data_max > 0:
                y_langau_map = y_langau_map / y_langau_map.max() * data_max
            ax.plot(
                x_eval,
                y_langau_map,
                color="darkorange",
                linestyle="-.",
                lw=2.0,
                label="LanGau approximation",
            )
            curve_max = max(curve_max, y_langau_map.max())

    # Set axes limits cleanly based on current bin boundaries
    ax.set_xlim(hist_xmin, hist_xmax)
    ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)

    ax.set_xlabel(r"$dE/dx$ [MeV/cm]")
    ax.set_ylabel(f"hits / {bin_width:.2f} MeV/cm")
    
    tpc_str = "TPCs Combined" if tpc == -1 else f"tpc {tpc}"
    ax.set_title(f"plane {plane}, {tpc_str}, {rr_min:g} <= rr < {rr_max:g} cm")

    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
    return fig

In [ ]:
import os

hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(2, 3), (6, 7), (15, 16), (30, 31)]

# Define output path
output_dir = "/exp/sbnd/data/users/lpelegri/TLEGraphs/dEdxPDF"
os.makedirs(output_dir, exist_ok=True)

plane = 2
df = hit_dfs[plane]

for rr_min, rr_max in rr_ranges:
    # Set dynamic x_limits based on range
    if (rr_min, rr_max) == (2, 3):
        x_limits = (1.5, 10)
    elif (rr_min, rr_max) == (6, 7):
        x_limits = (1.5, 8)
    elif (rr_min, rr_max) == (15, 16):
        x_limits = (1.5, 5)
    else:
        x_limits = (1.5, 4)

    # Plot 1: Full diagnostic
    fig1 = plot_slice_diagnostic(
        df,
        plane=plane,
        tpc=-1,
        rr_min=rr_min,
        rr_max=rr_max,
        hfit=hfit,
        pdg=pdg,
        conv_tf1_map=conv_tf1_map,
        conv_shifted_tf1_map=conv_shifted_tf1_map,
        langau_map=langau_map,
        x_limits=x_limits,
    )

    fname1 = f"diag_full_p2_tpc2_rr_{rr_min:g}_{rr_max:g}.png"
    fig1.savefig(
        os.path.join(output_dir, fname1), dpi=300, bbox_inches="tight"
    )

    # Plot 2: Theory only
    fig2 = plot_slice_diagnostic(
        df,
        plane=plane,
        tpc=-1,
        rr_min=rr_min,
        rr_max=rr_max,
        hfit=hfit,
        pdg=pdg,
        conv_tf1_map=conv_tf1_map,
        conv_shifted_tf1_map=conv_shifted_tf1_map,
        langau_map=langau_map,
        x_limits=x_limits,
        plot_only_theory=True,
    )

    fname2 = f"diag_theory_p2_tpc2_rr_{rr_min:g}_{rr_max:g}.png"
    fig2.savefig(
        os.path.join(output_dir, fname2), dpi=300, bbox_inches="tight"
    )

In [ ]:
def plot_mpv_vs_pitch(
    hfit,
    pdg,
    rr_center,
    mass=None,
    pitch_min=0.1,
    pitch_max=2.0,
    n_pitch=50,
):
    """Sweep pitch through build_theoretical_pdf and plot the resulting
    theoretical MPV (peak of the unconvolved PDF) as a function of pitch,
    at fixed rr_center. Useful for seeing how sensitive the theory MPV is
    to the pitch value -- e.g. checking whether the hardcoded pitch=0.32
    in get_conv_function_map is close enough to the data's actual pitch
    range to not matter, or whether it's a real source of MPV mismatch.
    """
    pitches = np.linspace(pitch_min, pitch_max, n_pitch)
    mpvs = []
    for p in pitches:
        pdf = build_theoretical_pdf(hfit, pdg, rr_center, p, mass=mass)
        mpvs.append(pdf.GetMaximumX())
    mpvs = np.array(mpvs)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(pitches, mpvs, marker="o", markersize=3, lw=1.5, color="darkorange")
    ax.axvline(0.32, color="gray", linestyle="--", lw=1,
               label="hardcoded C++ pitch (0.32)")
    ax.set_xlabel("pitch [cm]")
    ax.set_ylabel("theory MPV [MeV/cm]")
    ax.set_title(f"Theoretical MPV vs pitch (rr={rr_center:g} cm, pdg={pdg})")
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
    return fig, pitches, mpvs

In [ ]:
fig, pitches, mpvs = plot_mpv_vs_pitch(hfit, pdg=13, rr_center=10.5, mass=0.105)

In [ ]:
import os
import matplotlib.pyplot as plt

# Filter pitch values
filtered_pitch = df[(df["pitch"] >= 0) & (df["pitch"] <= 2)]["pitch"].dropna()

# 1. Store the figure reference in fig2
fig2 = plt.figure(figsize=(8, 5))

plt.hist(
    filtered_pitch,
    bins=100,
    range=(0.25, 2),
    color="#1f77b4",
    edgecolor="black",
    alpha=0.7,
)

plt.xlabel("Pitch [cm]", fontsize=12)
plt.ylabel("Counts", fontsize=12)
plt.title("Distribution of Hit Pitch Values", fontsize=14)
plt.xlim(0.25, 2)
plt.grid(axis="y", linestyle="--", alpha=0.5)

# 2. Adjust layout BEFORE saving
plt.tight_layout()

# 3. Save the figure
fname2 = "pitch_values.png"
fig2.savefig(os.path.join(output_dir, fname2), dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ===========================================================================
# 9. Shift diagnostics: (a) the mode-shift dx between the theoretical PDF's
#    MPV and its Gaussian-smeared convolution's MPV, and (b) the offset
#    between the raw histogram's peak and the theoretical MPV -- both
#    plotted vs rr_center, one point per rr slice.
# ===========================================================================
from scipy.signal import savgol_filter
from scipy.signal import fftconvolve


from scipy.stats import gaussian_kde

def robust_hist_max_kde(values, xmin, xmax, n_grid=2000, bw_method=None):
    """Alternative to robust_hist_max: find the peak (mode) via a Gaussian
    KDE of the raw sample instead of a smoothed histogram. Less prone to
    the mode-shift bias Savitzky-Golay smoothing introduces on skewed
    (Landau-like) peaks, since it doesn't rely on a locally-symmetric
    polynomial fit.

    bw_method: passed to scipy.stats.gaussian_kde (e.g. a float scale
    factor, or 'scott'/'silverman'). None uses scipy's default (Scott's
    rule) -- worth tuning if the peak still looks over/under-smoothed.
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 5:
        return np.nan

    kde = gaussian_kde(vals, bw_method=bw_method)
    x_grid = np.linspace(xmin, xmax, n_grid)
    density = kde(x_grid)
    i_max = int(np.argmax(density))

    # == Parabolic refinement around the grid maximum, same idea as
    # == robust_hist_max's sub-bin interpolation
    if 0 < i_max < len(density) - 1:
        y0, y1, y2 = density[i_max - 1], density[i_max], density[i_max + 1]
        denom = (y0 - 2.0 * y1 + y2)
        step = x_grid[1] - x_grid[0]
        delta = 0.5 * (y0 - y2) / denom if denom != 0 else 0.0
        delta = np.clip(delta, -1.0, 1.0)
        return x_grid[i_max] + delta * step
    return x_grid[i_max]

def convolved_pdf_curve(x_grid, pdf_vals, dx, sigma):
    """Theory PDF (x) zero-mean Gaussian(sigma), evaluated back on x_grid
    itself -- no amplitude scaling, since only the peak *location* is
    needed here, not the height."""
    x_centered = x_grid - x_grid[len(x_grid) // 2]
    kernel = gaussian_kernel(x_centered, sigma)
    kernel = kernel / (kernel.sum() * dx)
    conv = fftconvolve(pdf_vals, kernel, mode="same") * dx
    return conv


def conv_mpv_from_grid(x_grid, conv_vals, xmin=0.0, xmax=10.0):
    """Grid-search peak location of a convolved curve, restricted to
    [xmin, xmax] (mirrors robust_max_x_py's search window)."""
    mask = (x_grid >= xmin) & (x_grid <= xmax)
    idx = np.argmax(conv_vals[mask])
    return x_grid[mask][idx]


def compute_shift_diagnostics(
    df, plane, tpc, hfit, pdg, fit_params,
    dedx_col="dedx", rr_min=3.0, rr_max=40.0, rr_bin_width=1.0,
    pitch=0.55, mass=None, min_entries=200,
    hist_nbins=1000, hist_xmin=0.0, hist_xmax=10.0,
    savgol_window=51, savgol_poly=3,
    peak_method="savgol",   # == "savgol" or "kde"
    kde_bw_method=None,
    verbose=False,
):
    popt, _ = fit_params.get((plane, tpc), (None, None))
    if popt is None:
        raise ValueError(
            f"No sigma_G(MPV) power-law fit available for (plane={plane}, tpc={tpc}); "
            "run analyze(...)/analyze_theoretical(...) first."
        )

    sl_all = df if tpc == -1 else df[df["tpc"] == tpc]
    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []

    for lo, hi in zip(edges[:-1], edges[1:]):
        sl = sl_all[(sl_all["rr"] >= lo) & (sl_all["rr"] < hi)]
        if len(sl) < min_entries:
            continue
        rr_center = 0.5 * (lo + hi)

        pdf = build_theoretical_pdf(hfit, pdg, rr_center, pitch, mass=mass)
        theory_mpv = robust_max_x_py(pdf, 0.0, 10.0, 2000)
        x_grid, pdf_vals, dx = build_pdf_grid(pdf)

        # == Predicted smearing sigma at this slice's theory MPV, and the
        # == resulting convolution's MPV
        predicted_sigma_g = power_law(theory_mpv, *popt)
        conv_vals = convolved_pdf_curve(x_grid, pdf_vals, dx, predicted_sigma_g)
        conv_mpv = conv_mpv_from_grid(x_grid, conv_vals, xmin=0.0, xmax=10.0)
        dx_shift = conv_mpv - theory_mpv   # == was theory_mpv - conv_mpv

        if peak_method == "kde":
            hist_max = robust_hist_max_kde(
                sl[dedx_col].to_numpy(), hist_xmin, hist_xmax,
                bw_method=kde_bw_method,
            )
        else:
            hist_max, _, _, _ = robust_hist_max(
                sl[dedx_col].to_numpy(), hist_xmin, hist_xmax,
                nbins=hist_nbins, savgol_window=savgol_window, savgol_poly=savgol_poly,
            )
        diff_hist_theory = hist_max - theory_mpv

        if verbose:
            print(f"  rr={rr_center:.2f}: theory_mpv={theory_mpv:.3f}, "
                  f"sigma_G={predicted_sigma_g:.3f}, conv_mpv={conv_mpv:.3f}, "
                  f"dx_shift={dx_shift:.3f}, hist_max={hist_max:.3f}, "
                  f"diff_hist_theory={diff_hist_theory:.3f}")

        rows.append(dict(
            plane=plane, tpc=tpc, rr_center=rr_center, n_hits=len(sl),
            theory_mpv=theory_mpv, predicted_sigma_g=predicted_sigma_g,
            conv_mpv=conv_mpv, dx_shift=dx_shift,
            hist_max=hist_max, diff_hist_theory=diff_hist_theory,
        ))

    return pd.DataFrame(rows)
    
def exp_decay_plateau(rr, a, b, c):
    """f(rr) = a + b * exp(-rr / c) -- generic model for a quantity that
    starts at (a+b) near rr=0 and relaxes toward a plateau `a` at large rr,
    with `c` setting the decay length scale."""
    return a + b * np.exp(-rr / c)


def fit_diff_hist_theory(diag_df, x_col="rr_center", y_col="diff_hist_theory", p0=None):
    """Fits exp_decay_plateau to diff_hist_theory vs rr_center.
    Returns (popt, perr) or (None, None) if the fit fails or there
    aren't enough points."""
    xs = diag_df[x_col].to_numpy()
    ys = diag_df[y_col].to_numpy()
    mask = np.isfinite(xs) & np.isfinite(ys)
    xs, ys = xs[mask], ys[mask]

    if len(xs) < 4:
        return None, None

    if p0 is None:
        # == Rough starting guess: plateau ~ value at largest rr,
        # == amplitude ~ (value at smallest rr) - plateau, decay length
        # == ~ a third of the rr range
        order = np.argsort(xs)
        a0 = ys[order][-1]
        b0 = ys[order][0] - a0
        c0 = max((xs.max() - xs.min()) / 3.0, 1e-3)
        p0 = [a0, b0, c0]

    try:
        popt, pcov = curve_fit(exp_decay_plateau, xs, ys, p0=p0, maxfev=20000)
        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except Exception:
        return None, None

def plot_shift_diagnostics(diag_df, plane, tpc, pitch=None, out_prefix="shift_diag", fit_diff_hist_theory_curve=True):
    """Single-panel plot from compute_shift_diagnostics's output: dx_shift
    and diff_hist_theory overlaid on the same axes vs rr_center, since both
    are MPV-offset quantities in the same units.

    pitch: the pitch value theory_mpv/conv_mpv were computed at -- shown in
    the title and printed, purely for labeling.

    fit_diff_hist_theory_curve: if True, fits exp_decay_plateau to
    diff_hist_theory vs rr_center and overlays the fitted curve.
    """
    if pitch is not None:
        print(f"theoretical MPV computed at pitch = {pitch:.2f} cm")

    fig, ax = plt.subplots(figsize=(8, 5.5))

    ax.axhline(0, color="gray", lw=1, linestyle=":")
    ax.plot(
        diag_df["rr_center"], diag_df["dx_shift"], "o-",
        color="darkorange", ms=4,
        label=r"conv(theory, $\sigma_G$) $-$ theory  (mode-shift from smearing)",
    )
    ax.plot(
        diag_df["rr_center"], diag_df["diff_hist_theory"], "s-",
        color="mediumpurple", ms=4,
        label=r"hist. max $-$ theory  (data peak vs. unsmeared theory)",
    )

    if fit_diff_hist_theory_curve:
        popt, perr = fit_diff_hist_theory(diag_df)
        if popt is not None:
            a, b, c = popt
            a_e, b_e, c_e = perr
            print(f"diff_hist_theory fit: a={a:.4f}+/-{a_e:.4f}, "
                  f"b={b:.4f}+/-{b_e:.4f}, c={c:.4f}+/-{c_e:.4f}")
            xs_fit = np.linspace(diag_df["rr_center"].min(), diag_df["rr_center"].max(), 200)
            ys_fit = exp_decay_plateau(xs_fit, *popt)
            ax.plot(
                xs_fit, ys_fit, "--", color="mediumpurple", lw=1.8, alpha=0.8,
                label=rf"fit: ${a:.3f} + {b:.3f}\,e^{{-rr/{c:.3f}}}$",
            )
        else:
            print("diff_hist_theory fit failed or too few points -- skipping overlay")

    ax.set_xlabel("rr [cm]")
    ax.set_ylabel(r"$\Delta$ MPV [MeV/cm]")
    tpc_label = "combined" if tpc == -1 else tpc
    title = f"plane {plane}, tpc {tpc_label}"
    if pitch is not None:
        title += f", pitch={pitch:.2f} cm"
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle=":", alpha=0.5)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_p{plane}_t{tpc}.pdf")
    plt.show()
    return fig
    
def robust_hist_max(values, xmin, xmax, nbins=1000, savgol_window=51, savgol_poly=3):
    """Find the peak (mode) of a sample via a finely-binned histogram,
    smoothed with a Savitzky-Golay filter, then refined to sub-bin
    precision with parabolic interpolation around the smoothed maximum.

    Raw argmax on an O(1000)-bin histogram is dominated by Poisson noise
    bin-to-bin; smoothing first (rather than just coarsening the binning)
    keeps the underlying peak shape while suppressing that noise, and the
    parabolic refinement recovers precision the smoothing softens.
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    counts, edges = np.histogram(vals, bins=nbins, range=(xmin, xmax))
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_width = centers[1] - centers[0]

    # == Savgol window must be odd and <= number of bins, and > polyorder
    window = min(savgol_window, len(counts))
    if window % 2 == 0:
        window -= 1
    window = max(window, savgol_poly + 2 if (savgol_poly + 2) % 2 else savgol_poly + 3)

    if window >= len(counts) or window < 5:
        # == Not enough bins to smooth meaningfully -- fall back to raw argmax
        i_max = int(np.argmax(counts))
        return centers[i_max], centers, counts, counts.astype(float)

    smoothed = savgol_filter(counts.astype(float), window_length=window, polyorder=savgol_poly)
    i_max = int(np.argmax(smoothed))

    # == Parabolic (3-point) interpolation around the smoothed peak for
    # == sub-bin precision -- fits a parabola through (i-1, i, i+1) and
    # == returns the vertex position rather than just the discrete bin center.
    if 0 < i_max < len(smoothed) - 1:
        y0, y1, y2 = smoothed[i_max - 1], smoothed[i_max], smoothed[i_max + 1]
        denom = (y0 - 2.0 * y1 + y2)
        delta = 0.5 * (y0 - y2) / denom if denom != 0 else 0.0
        delta = np.clip(delta, -1.0, 1.0)
        x_max = centers[i_max] + delta * bin_width
    else:
        x_max = centers[i_max]

    return x_max, centers, counts, smoothed

In [ ]:
fit_params = {
    # Plane 0
    (0, 0): ([0.18325782, 0.0030161105, 3.2302234], None),
    (0, 1): ([0.18325782, 0.0030161105, 3.2302234], None),
    (0, -1): ([0.18325782, 0.0030161105, 3.2302234], None),
    # Plane 1
    (1, 0): ([0.23192165, 0.0080750843, 2.6270927], None),
    (1, 1): ([0.23192165, 0.0080750843, 2.6270927], None),
    (1, -1): ([0.23192165, 0.0080750843, 2.6270927], None),
    # Plane 2
    (2, 0): ([0.071890269, 0.0029818055, 3.3492475], None),
    (2, 1): ([0.071890269, 0.0029818055, 3.3492475], None),
    (2, -1): ([0.071890269, 0.0029818055, 3.3492475], None),
}

In [ ]:
tpc = -1

diag_df_032 = compute_shift_diagnostics(
    hit_dfs[plane], plane, tpc, hfit, pdg=pdg, fit_params=fit_params,
    pitch=0.32, verbose=False,
)
shift_fig = plot_shift_diagnostics(diag_df_032, plane, tpc, pitch=0.32, out_prefix="shift_diag_pitch032")

fname2 = "hist_maximum_theory_conv_theory.png"
shift_fig.savefig(os.path.join(output_dir, fname2), dpi=300, bbox_inches="tight")


diag_df_040 = compute_shift_diagnostics(
    hit_dfs[plane], plane, tpc, hfit, pdg=pdg, fit_params=fit_params,
    pitch=0.40, verbose=False,
)
plot_shift_diagnostics(diag_df_040, plane, tpc, pitch=0.40, out_prefix="shift_diag_pitch040")

diag_df_045 = compute_shift_diagnostics(
    hit_dfs[plane], plane, tpc, hfit, pdg=pdg, fit_params=fit_params,
    pitch=0.45, verbose=False,
)
plot_shift_diagnostics(diag_df_045, plane, tpc, pitch=0.45, out_prefix="shift_diag_pitch045")

diag_df_055 = compute_shift_diagnostics(
    hit_dfs[plane], plane, tpc, hfit, pdg=pdg, fit_params=fit_params,
    pitch=0.55, verbose=False,
)
plot_shift_diagnostics(diag_df_055, plane, tpc, pitch=0.55, out_prefix="shift_diag_pitch055")

In [ ]:
def plot_slice_diagnostic_w_pitch(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit=None,
    pdg=13,
    mass=None,
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=20.0,
    pitch_min=0.31,
    pitch_max=0.33,
    fit_params=None,
    first_stage_range=(0.0, 10.0),
    conv_tf1_map=None,
    conv_shifted_tf1_map=None,
    langau_map=None,
    conv_max_rr=1000,
    x_limits=None,
    plot_only_theory=False,
):
    # If x_limits is specified, use it to redefine the histogram boundaries
    if x_limits is not None:
        hist_xmin, hist_xmax = x_limits

    # Filter by TPC, Residual Range (rr), and specific Pitch window
    sl = df if tpc == -1 else df[df["tpc"] == tpc]
    sl = sl[
        (sl["rr"] >= rr_min)
        & (sl["rr"] < rr_max)
        & (sl["pitch"] >= pitch_min)
        & (sl["pitch"] <= pitch_max)
    ]
    rr_center = 0.5 * (rr_min + rr_max)

    i_rr_lookup = int(np.floor(rr_center))
    i_rr_lookup = max(0, min(i_rr_lookup, conv_max_rr - 1))
    rr_aligned = i_rr_lookup + 0.5

    # If plot_only_theory is True, disable all convolution/map fits
    if plot_only_theory:
        conv_tf1_map = None
        conv_shifted_tf1_map = None
        langau_map = None

    fig, ax = plt.subplots(figsize=(8, 5.5))
    counts, edges, _ = ax.hist(
        sl[dedx_col].dropna(),
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"Measured PDF (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 400)
    data_max = counts.max() if len(counts) else 0.0
    curve_max = data_max

    # Track curves for visible range scaling
    y_theo, y_conv, y_conv_shifted, y_langau_map = None, None, None, None

    hist = th1_from_series(
        sl[dedx_col],
        f"diag_p{plane}_t{tpc}",
        "",
        nbins,
        hist_xmin,
        hist_xmax,
    )

    if hfit is not None:
        mean_pitch = 0.5 * (pitch_min + pitch_max)
        
        # Pass fit_params to build_theoretical_pdf if provided
        pdf_kwargs = {"mass": mass}
        if fit_params is not None:
            pdf_kwargs["fit_params"] = fit_params

        pdf = build_theoretical_pdf(hfit, pdg, rr_aligned, mean_pitch, **pdf_kwargs)
        norm = len(sl) * bin_width
        y_theo = np.array([pdf.Eval(xv) * norm for xv in x_eval])
        if y_theo.max() > 0 and data_max > 0:
            y_theo = y_theo / y_theo.max() * data_max
        ax.plot(
            x_eval,
            y_theo,
            color="#785EF0",  # Colorblind-friendly purple
            linestyle=":",
            lw=2.5,
            label="Theoretical PDF",
        )
        curve_max = max(curve_max, y_theo.max())

    if conv_tf1_map is not None:
        f_conv = get_conv_tf1_for_slice(conv_tf1_map, plane, rr_center, conv_max_rr)
        if f_conv is not None:
            y_conv = np.array([f_conv.Eval(xv) for xv in x_eval])
            if y_conv.max() > 0 and data_max > 0:
                y_conv = y_conv / y_conv.max() * data_max
            ax.plot(
                x_eval,
                y_conv,
                color="crimson",
                linestyle="-.",
                lw=2.0,
                label="Convolution",
            )
            curve_max = max(curve_max, y_conv.max())

    if conv_shifted_tf1_map is not None:
        f_conv_shifted = get_conv_tf1_for_slice(
            conv_shifted_tf1_map, plane, rr_center, conv_max_rr
        )
        if f_conv_shifted is not None:
            y_conv_shifted = np.array([f_conv_shifted.Eval(xv) for xv in x_eval])
            if y_conv_shifted.max() > 0 and data_max > 0:
                y_conv_shifted = y_conv_shifted / y_conv_shifted.max() * data_max
            ax.plot(
                x_eval,
                y_conv_shifted,
                color="teal",
                linestyle="-.",
                lw=2.0,
                label="Convolution (shifted)",
            )
            curve_max = max(curve_max, y_conv_shifted.max())

    if langau_map is not None:
        f_langau_map = get_conv_tf1_for_slice(langau_map, plane, rr_center, conv_max_rr)
        if f_langau_map is not None:
            y_langau_map = np.array([f_langau_map.Eval(xv) for xv in x_eval])
            if y_langau_map.max() > 0 and data_max > 0:
                y_langau_map = y_langau_map / y_langau_map.max() * data_max
            ax.plot(
                x_eval,
                y_langau_map,
                color="darkorange",
                linestyle="-.",
                lw=2.0,
                label="LanGau approximation",
            )
            curve_max = max(curve_max, y_langau_map.max())

    # Set axes limits cleanly based on current bin boundaries
    ax.set_xlim(hist_xmin, hist_xmax)
    ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)

    ax.set_xlabel(r"$dE/dx$ [MeV/cm]")
    ax.set_ylabel(f"hits / {bin_width:.2f} MeV/cm")

    tpc_str = "TPCs Combined" if tpc == -1 else f"tpc {tpc}"
    ax.set_title(
        f"plane {plane}, {tpc_str}, {rr_min:g} <= rr < {rr_max:g} cm, "
        f"{pitch_min:g} <= pitch <= {pitch_max:g} cm"
    )

    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
    return fig

In [ ]:
fig2 = plot_slice_diagnostic_w_pitch(
    df,
    plane,
    -1,
    2,
    3,
    hfit=hfit,
    pdg=pdg,
    dedx_col="dedx",
    nbins=100,
    hist_xmin=1.5,
    hist_xmax=7.0,
    pitch_min=0.5,
    pitch_max=0.6,
    plot_only_theory=True,
)